# Runway Screening — Production Training on Colab (A100)

Trains **YOLOv8m** on merged RDD2022 + verified drone labels at **imgsz 960** — the config that forces batch-1 on a 6 GB laptop GPU but runs comfortably on an A100.

**Before you run:** `Runtime → Change runtime type → A100 GPU` (needs Colab Pro/Pro+).
Then run every cell top-to-bottom. Setup (clone + 13 GB RDD download + merge) is ~10–15 min; training is a few hours.

The dataset is rebuilt from source here (not uploaded) so the split is identical to local — `merge_datasets.py` uses a fixed seed.

## 1. Confirm the A100 is attached

In [ ]:
!nvidia-smi

## 2. Install ultralytics

In [ ]:
!pip -q install ultralytics
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))

## 3. Clone the repo (branch `local/nano-bootstrap`)
Brings in `src/`, the 274 drone `frames/`, and the 154 verified `drone_labels_v1/`.

In [ ]:
!git clone --branch local/nano-bootstrap --single-branch https://github.com/shabazahmadme1-eng/runway_screening.git
%cd runway_screening

## 4. Download RDD2022 from FigShare (~13 GB, no auth)
Official CRDDC2022 release. `wget -c` resumes if the runtime hiccups.

In [ ]:
import os
os.makedirs('data/rdd2022', exist_ok=True)
!wget -c -O data/rdd2022/RDD2022_CRDDC.zip https://ndownloader.figshare.com/files/38030910

## 5. Extract (outer zip + the 7 nested per-country zips)
Uses Python `zipfile` — bsdtar chokes on this archive.

In [ ]:
import zipfile
from pathlib import Path

ZIP = Path('data/rdd2022/RDD2022_CRDDC.zip')
DEST = Path('data/rdd2022/extracted')
DEST.mkdir(parents=True, exist_ok=True)

print('extracting outer zip ...', flush=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall(DEST)
for nz in sorted(DEST.rglob('*.zip')):
    out = nz.parent / nz.stem
    print('extracting nested', nz.name, flush=True)
    with zipfile.ZipFile(nz) as z:
        z.extractall(out)
    nz.unlink()  # free disk as we go
print('xml:', sum(1 for _ in DEST.rglob('*.xml')), '| jpg:', sum(1 for _ in DEST.rglob('*.jpg')))

## 6. Build the master split (RDD2022 + verified drone labels)
Should report `nc: 12` and ~21k train / ~5.3k val.

In [ ]:
!python src/merge_datasets.py --drone-frames frames --drone-labels drone_labels_v1 --out datasets/master --rdd-dir data/rdd2022/extracted
print('\n--- data.yaml ---')
print(open('datasets/master/data.yaml').read())

## 7. Train — YOLOv8m, 150 epochs @ 960
`--batch -1` auto-sizes to the A100 (expect a big batch). On the A100 this is a few hours; `patience=50` may early-stop sooner.

In [ ]:
!python src/train.py --model yolov8m.pt --data datasets/master/data.yaml --epochs 150 --imgsz 960 --batch -1 --workers 8 --name runway_m_v1

## 8. Save artifacts to Google Drive
Copies `best.pt` (as `yolov8m_v1.pt`), `results.csv`, and `confusion_matrix.png` to `MyDrive/runway_m_v1/` so they survive the runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil

run = 'runs/detect/runway_m_v1'
dst = '/content/drive/MyDrive/runway_m_v1'
os.makedirs(dst, exist_ok=True)
shutil.copy(f'{run}/weights/best.pt', f'{dst}/yolov8m_v1.pt')
for fn in ['results.csv', 'confusion_matrix.png', 'args.yaml']:
    p = f'{run}/{fn}'
    if os.path.exists(p):
        shutil.copy(p, dst)
print('saved to', dst, '->', sorted(os.listdir(dst)))
print('best.pt size MB:', round(os.path.getsize(f'{dst}/yolov8m_v1.pt') / 1e6, 1))

## 9. (Optional) Push artifacts straight back to GitHub
yolov8m `best.pt` is ~50 MB — under GitHub's 100 MB limit. Paste a [Personal Access Token](https://github.com/settings/tokens) (repo scope) and run. Otherwise skip and pull from Drive locally.

In [ ]:
TOKEN = ''  # <-- paste a GitHub PAT with 'repo' scope, then run this cell

if TOKEN:
    import os, shutil
    run = 'runs/detect/runway_m_v1'
    os.makedirs('weights', exist_ok=True)
    os.makedirs('reports/runway_m_v1', exist_ok=True)
    shutil.copy(f'{run}/weights/best.pt', 'weights/yolov8m_v1.pt')
    for fn in ['results.csv', 'confusion_matrix.png']:
        if os.path.exists(f'{run}/{fn}'):
            shutil.copy(f'{run}/{fn}', 'reports/runway_m_v1/')
    !git config user.email "neuroparadigm@gmail.com"
    !git config user.name "Shabaz"
    !git add -f weights/yolov8m_v1.pt reports/runway_m_v1
    !git -c commit.gpgsign=false commit -m "Add yolov8m production weights + reports (Colab A100, 960px)"
    !git remote set-url origin https://{TOKEN}@github.com/shabazahmadme1-eng/runway_screening.git
    !git push origin local/nano-bootstrap
else:
    print('No token set — skipping push. Artifacts are in Drive from cell 8.')